# Ad-Dhakhira sur Kaggle

Assistant de recherche bibliographique dans le fiqh mālikite, lancé dans une session Kaggle gratuite (2 GPU T4) et partagé par un lien public, protégé par identifiant et mot de passe.

## Avant le premier lancement (une seule fois)

1. **Compte Kaggle vérifié par téléphone** : *Settings → Phone verification*. Sans vérification, pas de GPU ni d'accès Internet.
2. Dans ce notebook, panneau de droite **Session options** :
   - **Accelerator** : `GPU T4 x2` ;
   - **Internet** : activé (`On`).
3. Menu **Add-ons → Secrets** : ajoute les secrets utiles, et coche-les pour ce notebook.

| Secret | Rôle |
|---|---|
| `APP_USERS` | **Recommandé.** Comptes autorisés, séparés par des virgules : `samy:MotDePasse1,fatima:MotDePasse2` |
| `GEMINI_API_KEY` | Facultatif. Clé Gemini fournie à tous les utilisateurs |
| `OPENAI_API_KEY`, `ANTHROPIC_API_KEY` | Facultatifs. Idem pour ChatGPT et Claude |
| `HF_TOKEN` | Facultatif. Jeton Hugging Face (téléchargements plus rapides) |

Sans aucune clé API, l'outil fonctionne quand même avec le **moteur local** (Qwen2.5 7B sur le premier GPU). Chaque utilisateur peut aussi ajouter ses propres clés dans l'onglet *Paramètres* de l'interface.

## Lancer une session

- **Mode interactif** : *Run All*. Le lien public s'affiche à la fin de l'étape 5. Garde l'onglet ouvert : Kaggle coupe une session interactive inactive.
- **Mode arrière-plan** (recommandé pour partager longtemps) : *Save Version → Save & Run All (Commit)*. La session tourne sans navigateur ouvert, pendant `SESSION_HOURS` heures (4 par défaut, 12 au maximum). Le lien apparaît dans le **journal** de la version en cours (*View Active Events → Logs*), rappelé toutes les 10 minutes.

Compte 10 à 15 minutes de démarrage : installation, puis téléchargement des modèles. Le lien change à chaque session.

## 1. Réglages

In [ ]:
REPO_URL = "https://github.com/SamyTZ5/AdDhakhiraCorpusAI_vIhsan.git"
REPO_BRANCH = "main"

# Moteur local Qwen2.5-7B (sans clé API) sur le premier GPU. Installation plus longue.
# Si l'étape 2 signale un problème avec vLLM, mets False puis « Run → Restart & Run All ».
ENABLE_LOCAL_ENGINES = True

# Durée pendant laquelle le lien reste ouvert (Kaggle arrête toute session à 12 h).
# La session consomme le quota GPU hebdomadaire (30 h) pendant toute cette durée.
SESSION_HOURS = 4

WORK_DIR = "/kaggle/working/addhakhira"   # code (léger)
DATA_DIR = "/tmp/addhakhira"              # modèles et index (non conservés entre sessions)

## 2. Installation

In [ ]:
import os, subprocess, sys
from pathlib import Path


def run(cmd, cwd=None):
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if proc.returncode != 0:
        print("\n".join(proc.stdout.strip().splitlines()[-40:]), flush=True)
        raise RuntimeError(f"Échec de : {' '.join(cmd)} (voir les lignes ci-dessus)")
    return proc.stdout


print(run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"]))
repo = Path(WORK_DIR)
if (repo / ".git").exists():
    run(["git", "fetch", "--depth", "1", "origin", REPO_BRANCH], cwd=repo)
    run(["git", "reset", "--hard", f"origin/{REPO_BRANCH}"], cwd=repo)
else:
    run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(repo)])
print("Version du code :", run(["git", "log", "--oneline", "-1"], cwd=repo).strip())

# Bibliothèques de l'outil. PyTorch est déjà fourni par Kaggle.
run([sys.executable, "-m", "pip", "install", "-q",
     "transformers==5.10.1", "sentence-transformers==5.3.0", "huggingface-hub==1.16.1",
     "tokenizers==0.22.2", "safetensors==0.6.2", "accelerate==1.7.0", "sentencepiece==0.2.1",
     "tiktoken==0.8.0", "faiss-cpu==1.13.2", "rank-bm25==0.2.2", "google-genai==2.10.0",
     "openai==2.28.0", "anthropic==0.82.0", "gradio==6.15.1"])

LOCAL_READY = False
if ENABLE_LOCAL_ENGINES:
    print("Installation de vLLM pour le moteur local (quelques minutes)…", flush=True)
    try:
        run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.24.0"])
        check = run([sys.executable, "-c", "import torch, vllm; print(torch.__version__, torch.cuda.is_available())"])
        LOCAL_READY = check.strip().endswith("True")
        print("Moteur local :", "prêt" if LOCAL_READY else "GPU non détecté après l'installation de vLLM")
    except RuntimeError as exc:
        print("Moteur local indisponible :", exc)
    if not LOCAL_READY:
        print("→ Mets ENABLE_LOCAL_ENGINES = False puis « Run → Restart & Run All » pour une session sans moteur local.")
print("Installation terminée.")

## 3. Modèles et index

In [ ]:
os.environ["HF_HOME"] = f"{DATA_DIR}/hf_cache"
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
except Exception:
    _secrets = None


def secret(name):
    if _secrets is None:
        return ""
    try:
        return (_secrets.get_secret(name) or "").strip()
    except Exception:
        return ""


if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")

from huggingface_hub import hf_hub_download, snapshot_download

models = Path(DATA_DIR) / "models"
embedding_dir = models / "Qwen__Qwen3-Embedding-4B"
index_root = Path(DATA_DIR) / "vector_indexes"
lite_dir = models / "Qwen__Qwen2.5-7B-Instruct-AWQ"

print("Modèle de recherche (environ 8 Go)…", flush=True)
snapshot_download("Qwen/Qwen3-Embedding-4B", local_dir=str(embedding_dir))
for filename in ("signature.json", "index.faiss"):
    hf_hub_download("userzh92/addhakhira-faiss-index", repo_type="dataset",
                    filename=f"Qwen__Qwen3-Embedding-4B/{filename}",
                    local_dir=str(index_root / "faiss"))
print("Index prêt.")
if LOCAL_READY:
    print("Moteur local Qwen2.5-7B (environ 5,5 Go)…", flush=True)
    snapshot_download("Qwen/Qwen2.5-7B-Instruct-AWQ", local_dir=str(lite_dir))
print("Téléchargements terminés.")

## 4. Configuration

In [ ]:
for name in ("APP_USERS", "APP_PASSWORD", "GEMINI_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY",
             "GEMINI_MODEL", "OPENAI_MODEL", "ANTHROPIC_MODEL"):
    value = secret(name)
    if value:
        os.environ[name] = value

gpu_count = len([l for l in run(["nvidia-smi", "-L"]).splitlines() if l.startswith("GPU")])
os.environ["ADDHAKHIRA_HOSTING"] = "kaggle"
# Deux GPU : le modèle de recherche reste chargé sur le second, le moteur local utilise le premier.
os.environ["ADDHAKHIRA_EMBEDDING_DEVICE"] = "cuda:1" if gpu_count >= 2 else ""

template = (repo / "src" / "config_template.py").read_text(encoding="utf-8")
overrides = f'''

# --- Réglages Kaggle (générés par le notebook) ---
LLM_BACKEND = "gemini_api"
GEMINI_API_KEY = ""  # les clés restent dans les variables d'environnement
MODEL_EXTRACTOR_PATH = "{models / 'non-disponible'}"
MODEL_REASONER_PATH = "{models / 'non-disponible'}"
LITE_MODEL_PATH = "{lite_dir if LOCAL_READY else ''}"
EMBEDDING_MODEL = "{embedding_dir}"
EMBEDDING_INDEX_DIR = Path("{index_root}")
ENABLE_DENSE_RETRIEVAL = True
VECTOR_INDEX_BACKEND = "faiss"
NUM_GPUS_EXTRACTOR = 1
NUM_GPUS_REASONER = 1
'''
(repo / "src" / "config.py").write_text(template + overrides, encoding="utf-8")
print("GPU :", gpu_count, "| moteur local :", "oui" if LOCAL_READY else "non",
      "| comptes :", "oui" if os.environ.get("APP_USERS") or os.environ.get("APP_PASSWORD") else "AUCUN (accès libre)",
      "| clés serveur :", ", ".join(n for n in ("GEMINI_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY") if os.environ.get(n)) or "aucune")

## 5. Lancement et lien public

In [ ]:
import time

if not (Path(WORK_DIR) / "src" / "config.py").exists():
    raise RuntimeError(
        "Les étapes 2 à 4 n'ont pas été exécutées jusqu'au bout. Remonte pour voir la première erreur, "
        "puis relance avec « Run All »."
    )
os.chdir(repo)
sys.path.insert(0, str(repo))
from src.server import build_app, open_public_link, serve_in_background

app = build_app()
serve_in_background(app, port=7860)
url = open_public_link(7860)
banner = "=" * 70
print(f"{banner}\nLIEN À PARTAGER : {url}\n{banner}", flush=True)
print("Les modèles finissent de se charger en arrière-plan : l'interface l'indique par un bandeau.")

deadline = time.time() + SESSION_HOURS * 3600
while time.time() < deadline:
    time.sleep(600)
    remaining = (deadline - time.time()) / 3600
    print(f"[{time.strftime('%H:%M')}] En ligne : {url} (encore {remaining:.1f} h)", flush=True)